# Kuantisasi dan Benchmarking Model YOLO11n VNetra
Notebook ini didedikasikan khusus untuk berjalan di **Google Colab TPU**.

Fungsi utama notebook ini adalah:
1. Mengambil bobot asli `.pt` dari Google Drive (hasil training Kaggle).
2. Melakukan kompresi/kuantisasi model ke FP16 dan INT8 menggunakan spesifikasi Host CPU raksasa milik mesin TPU Colab.
3. Melakukan evaluasi otomatis (Akurasi, Kecepatan, Ukuran) pada ketiga varian model tersebut.
4. Membangun grafik profesional (Standar Jurnal/Skripsi) dan menyimpannya ke Google Drive.

## 1. Persiapan Lingkungan dan Google Drive
Pastikan Anda telah mengubah Runtime mesin ke **TPU** (*Runtime -> Change runtime type -> TPU*).

In [ ]:
import IPython
import PIL
pil_ver = PIL.__version__
IPython.get_ipython().system(f"pip install ultralytics tensorflow seaborn matplotlib pandas Pillow=={pil_ver}")

import importlib
import site
importlib.reload(site)
importlib.invalidate_caches()
from google.colab import drive
drive.mount('/content/drive')
import os

# --- KONFIGURASI EKSPERIMEN (BISA DIUBAH) ---
EXPERIMENT_ID = 1

DRIVE_BASE_DIR = f'/content/drive/MyDrive/YOLO/eksperimen_{EXPERIMENT_ID}'
INPUT_DIR = f'{DRIVE_BASE_DIR}/input'
OUTPUT_DIR = f'{DRIVE_BASE_DIR}/output'

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Environment siap!")


## 2. Pemulihan Dataset dan Model dari Drive
Mengekstrak `vnetra_master_dataset.zip` yang wajib Anda letakkan di root Google Drive, serta meload model FP32 `best_yolo11n.pt`.

In [ ]:
master_dir = "/content/vnetra_master_dataset"
model_pt_path = f"{OUTPUT_DIR}/best.pt"
dataset_zip_path = f"{INPUT_DIR}/vnetra_master_dataset.zip"

if not os.path.exists(model_pt_path):
    raise FileNotFoundError("ERROR: File best_yolo11n.pt tidak ditemukan di MyDrive. Pastikan Anda sudah menguploadnya dari hasil Kaggle!")

if not os.path.exists(dataset_zip_path):
    raise FileNotFoundError("ERROR: File vnetra_master_dataset.zip tidak ditemukan di MyDrive!")

if not os.path.exists(master_dir):
    print("Mengekstrak dataset dari Google Drive, mohon tunggu...")
    os.system(f'unzip -q {dataset_zip_path} -d /content/')
    print("Ekstraksi dataset selesai!")
else:
    print("Dataset sudah diekstrak sebelumnya.")

from ultralytics import YOLO
model = YOLO(model_pt_path)
print("Model FP32 asli berhasil dimuat!")


In [ ]:
import os
import pandas as pd

master_classes = [
    'person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'train', 
    'stop sign', 'bench', 'chair', 'potted plant', 'dog', 'cat',
    'pothole', 'open_drain', 'puddle', 'pole', 
    'hanging_branch', 'tactile_paving_straight', 'tactile_paving_turn', 
    'tactile_paving_3way', 'tactile_paving_4way', 'tactile_paving_stop', 
    'stairs_up', 'stairs_down', 'curb', 'crosswalk', 'tree', 'fence'
]

print("\n=======================================================")
print("      LAPORAN AKHIR DISTRIBUSI DATASET VNETRA")
print("=======================================================")

def count_images(directory):
    if not os.path.exists(directory): return 0
    return len([f for f in os.listdir(directory) if f.endswith(('.jpg', '.jpeg', '.png'))])

train_count = count_images('vnetra_master_dataset/train/images')
valid_count = count_images('vnetra_master_dataset/valid/images')
test_count  = count_images('vnetra_master_dataset/test/images')
total_images = train_count + valid_count + test_count

if total_images > 0:
    print(f"Total Lembar Gambar (Images) Keseluruhan : {total_images} lembar")
    print(f"- Data Latih (Train)     : {train_count} lembar ({train_count/total_images*100:.2f}%)")
    print(f"- Data Ujian (Valid)     : {valid_count} lembar ({valid_count/total_images*100:.2f}%)")
    print(f"- Data Buta (Test)       : {test_count} lembar ({test_count/total_images*100:.2f}%)")
else:
    print("Belum ada gambar yang diproses.")

print("\n--- RINCIAN INSTANS (JUMLAH OBJEK / BOUNDING BOX) PER KELAS ---")
print("Catatan: Tabel di bawah ini murni menghitung jumlah instans objek di dalam gambar, BUKAN jumlah lembar gambar.")
def count_instances_per_class(label_dir, num_classes):
    counts = {i: 0 for i in range(num_classes)}
    if not os.path.exists(label_dir): return counts
    
    for lbl_file in os.listdir(label_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(label_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    if cls_id in counts:
                        counts[cls_id] += 1
    return counts

train_cls = count_instances_per_class('vnetra_master_dataset/train/labels', len(master_classes))
valid_cls = count_instances_per_class('vnetra_master_dataset/valid/labels', len(master_classes))
test_cls  = count_instances_per_class('vnetra_master_dataset/test/labels', len(master_classes))

data_report = []
for i, cls_name in enumerate(master_classes):
    data_report.append({
        'ID': i,
        'Kelas': cls_name,
        'Train': train_cls[i],
        'Valid': valid_cls[i],
        'Test': test_cls[i]
    })

df_report = pd.DataFrame(data_report)
print("\n[INFO] Semua kelas di atas telah lolos jaring pengaman (Minimal 50 instans di Valid Set). Aman untuk lanjut training!")



## 3. Ekspor ke FP16 (Half Precision)
Kompresi format standar untuk akselerasi Mobile GPU.

In [ ]:
print("Mengekspor model ke FP16...")
export_fp16 = model.export(format="tflite", half=True, optimize=True)

import shutil
fp16_drive_path = f'{OUTPUT_DIR}/best_fp16.tflite'
shutil.copy(export_fp16, fp16_drive_path)
print(f"Model FP16 berhasil disimpan di: {fp16_drive_path}")


## 4. Ekspor ke INT8 (Full Integer Quantization)
Memanfaatkan CPU raksasa dari Host VM TPU untuk menyelesaikan kalibrasi Representative Dataset secepat mungkin.

In [ ]:
print("Mengekspor model ke INT8 murni menggunakan CPU TPU...")
print("Kalibrasi dataset akan berlangsung (Tunggu hingga proses selesai)...")

export_int8 = model.export(format="tflite", int8=True, data=f"{master_dir}/data.yaml", optimize=True, device='cpu')

int8_drive_path = f'{OUTPUT_DIR}/best_int8.tflite'
shutil.copy(export_int8, int8_drive_path)
print(f"Model INT8 berhasil disimpan di: {int8_drive_path}")


## 5. Benchmarking Otomatis
Menguji model asli (.pt), FP16 (.tflite), dan INT8 (.tflite) menggunakan validation set untuk mendapatkan skor akurasi (mAP) dan latensi/kecepatan.

In [ ]:
print("\n=== EVALUASI MODEL FP32 ASLI (.pt) ===")
val_pt = model.val(data=f"{master_dir}/data.yaml", split='test')
map50_pt = val_pt.box.map50
map5095_pt = val_pt.box.map
speed_pt = sum(val_pt.speed.values()) # Total MS (Preprocess + Inference + Postprocess)
size_pt = os.path.getsize(model_pt_path) / (1024 * 1024)


In [ ]:
print("\n=== EVALUASI MODEL FP16 (.tflite) ===")
model_fp16 = YOLO(export_fp16, task='detect')
val_fp16 = model_fp16.val(data=f"{master_dir}/data.yaml", split='test')
map50_fp16 = val_fp16.box.map50
map5095_fp16 = val_fp16.box.map
speed_fp16 = sum(val_fp16.speed.values())
size_fp16 = os.path.getsize(export_fp16) / (1024 * 1024)


In [ ]:
print("\n=== EVALUASI MODEL INT8 (.tflite) ===")
model_int8 = YOLO(export_int8, task='detect')
val_int8 = model_int8.val(data=f"{master_dir}/data.yaml", split='test')
map50_int8 = val_int8.box.map50
map5095_int8 = val_int8.box.map
speed_int8 = sum(val_int8.speed.values())
size_int8 = os.path.getsize(export_int8) / (1024 * 1024)


## 6. Visualisasi Grafik Profesional (Untuk Laporan Skripsi)
Mengubah data dari hasil pengujian di atas menjadi grafik *Bar Chart* yang sangat rapi dan siap *copy-paste* ke Microsoft Word.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Menyusun data
data = {
    'Format': ['YOLO11n FP32 (Asli)', 'YOLO11n FP16', 'YOLO11n INT8'],
    'mAP@50': [map50_pt, map50_fp16, map50_int8],
    'Speed (ms)': [speed_pt, speed_fp16, speed_int8],
    'Size (MB)': [size_pt, size_fp16, size_int8]
}
df = pd.DataFrame(data)

sns.set_theme(style='whitegrid', context='talk')
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('Hasil Benchmarking Kuantisasi Model VNetra', fontsize=26, fontweight='bold', y=1.05)

# 1. Akurasi (mAP@50)
ax1 = sns.barplot(x='Format', y='mAP@50', data=df, ax=axes[0], palette='Blues_d')
axes[0].set_title('Akurasi (mAP@50) \u2191', fontsize=20, fontweight='bold')
axes[0].set_ylabel('mAP', fontsize=16)
axes[0].set_ylim(0, max(df['mAP@50']) * 1.15)
for p in ax1.patches:
    ax1.annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=15, fontweight='bold', xytext=(0, 5), textcoords='offset points')

# 2. Kecepatan (Inference Speed)
ax2 = sns.barplot(x='Format', y='Speed (ms)', data=df, ax=axes[1], palette='Reds_d')
axes[1].set_title('Kecepatan Latensi (ms/Gambar) \u2193', fontsize=20, fontweight='bold')
axes[1].set_ylabel('Milliseconds (Lebih rendah lebih baik)', fontsize=16)
axes[1].set_ylim(0, max(df['Speed (ms)']) * 1.15)
for p in ax2.patches:
    ax2.annotate(f"{p.get_height():.1f} ms", (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=15, fontweight='bold', xytext=(0, 5), textcoords='offset points')

# 3. Ukuran Model
ax3 = sns.barplot(x='Format', y='Size (MB)', data=df, ax=axes[2], palette='Greens_d')
axes[2].set_title('Ukuran File Model (MB) \u2193', fontsize=20, fontweight='bold')
axes[2].set_ylabel('Megabytes (Lebih rendah lebih baik)', fontsize=16)
axes[2].set_ylim(0, max(df['Size (MB)']) * 1.15)
for p in ax3.patches:
    ax3.annotate(f"{p.get_height():.1f} MB", (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=15, fontweight='bold', xytext=(0, 5), textcoords='offset points')

plt.tight_layout()

graph_path = f'{OUTPUT_DIR}/quantization_benchmark_results.png'
plt.savefig(graph_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n\u2705 Selesai! Grafik resolusi tinggi telah berhasil disimpan ke Google Drive Anda di: {graph_path}")
print("Anda dapat langsung melampirkannya ke dalam naskah Laporan Skripsi.")


## 7. Visualisasi Prediksi Langsung (Bounding Box)
Mengambil satu gambar acak dari dataset `valid` atau `test` yang ada di memori Colab (yang baru diekstrak), kemudian menjalankan deteksi langsung pada FP32 Asli dan INT8 untuk membandingkan seberapa akurat kotak deteksi (Bounding Box) visualnya sebelum Anda melampirkannya ke laporan skripsi.

In [ ]:
import random
import glob
import cv2
import matplotlib.pyplot as plt

# Mencari gambar acak dari dataset yang ada di Colab
test_images = glob.glob(f"{master_dir}/test/images/*.jpg")
if not test_images:
    test_images = glob.glob(f"{master_dir}/valid/images/*.jpg")

if test_images:
    test_img = random.choice(test_images)
    print(f"Menguji visualisasi gambar: {test_img}")
    
    # Prediksi menggunakan FP32 (Asli)
    res_pt = model.predict(source=test_img, imgsz=640)
    img_pt = res_pt[0].plot()  # Menggambar Bounding Box
    
    # Prediksi menggunakan INT8
    res_int8 = model_int8.predict(source=test_img, imgsz=640)
    img_int8 = res_int8[0].plot()  # Menggambar Bounding Box
    
    # Menampilkan Perbandingan Bersebelahan
    fig, ax = plt.subplots(1, 2, figsize=(16, 8))
    fig.suptitle('Perbandingan Visual Deteksi: FP32 Asli vs INT8 TFLite', fontsize=20, fontweight='bold', y=1.02)
    
    ax[0].imshow(cv2.cvtColor(img_pt, cv2.COLOR_BGR2RGB))
    ax[0].set_title("Prediksi Model Asli FP32 (.pt)", fontsize=16)
    ax[0].axis("off")
    
    ax[1].imshow(cv2.cvtColor(img_int8, cv2.COLOR_BGR2RGB))
    ax[1].set_title("Prediksi Kuantisasi INT8 (.tflite)", fontsize=16)
    ax[1].axis("off")
    
    plt.tight_layout()
    predict_graph_path = f'{OUTPUT_DIR}/visual_predict_comparison.png'
    plt.savefig(predict_graph_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n[V] Gambar visualisasi ini telah otomatis disimpan ke: {predict_graph_path}")
else:
    print("\u26a0\ufe0f Tidak ada gambar tes yang ditemukan di folder dataset.")
